# The dumb zone, measured

**Same fact. Same question. We only add more text around it.**

An open 7B model (Qwen2.5-7B-Instruct, 32K-token window) reads a public-domain novel with one fact
hidden inside. We grow the novel from 1K to 32K tokens and ask the same question every time.

Two kinds of question:
- **Find the words.** The question repeats the fact's words.
  *"What is the Wi-Fi password at the venue?"* ← *"The Wi-Fi password at the venue is blue-dorayaki-42."*
- **Find the meaning.** No shared words; you need to know where the landmark is.
  *"Which character has been to Hyderabad?"* ← *"Kabir's flat looks straight out at the Charminar."*

The second kind is how NoLiMa (Adobe, ICML 2025) tests long context, and it's closer to real work:
your agent has to connect *"the payment bug"* to `checkout.ts`.

In [ ]:
# Setup: libraries and the GPU
import os, re, glob, json, time, random, datetime, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
warnings.filterwarnings("ignore")

GPU = ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))
print("GPU:", GPU or "none", "| torch", torch.__version__, "| transformers", transformers.__version__)
assert GPU, "Turn on a GPU: Settings → Accelerator"
TODAY = datetime.date.today().isoformat()

In [ ]:
# Plot style: dark, big, readable from the back of a hall
BG, FG, MUTED, GRID = "#0f0f10", "#f5f5f2", "#9a9aa0", "#2b2b30"
RED, BLUE, GREEN, AMBER = "#ff5a5f", "#4c8dff", "#3ecf8e", "#ffb020"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": FG, "axes.labelcolor": FG, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "grid.color": GRID, "font.size": 16, "axes.titlesize": 24,
    "axes.titleweight": "bold", "axes.labelsize": 18, "xtick.labelsize": 16, "ytick.labelsize": 16,
    "legend.frameon": False, "axes.spines.top": False, "axes.spines.right": False,
})
LABEL = {1024: "1K", 2048: "2K", 4096: "4K", 8192: "8K", 16384: "16K", 31744: "32K"}

def spread(ys, gap, lo=-np.inf, hi=np.inf):
    # Nudge label heights apart, keeping their order, so line labels never sit on top of each other
    ys = np.asarray(ys, float)
    order = np.argsort(ys, kind="stable")
    out = ys[order].copy()
    for _ in range(200):
        for k in range(1, len(out)):
            if out[k] - out[k - 1] < gap:
                mid = (out[k] + out[k - 1]) / 2
                out[k - 1], out[k] = mid - gap / 2, mid + gap / 2
    out += min(0, hi - out[-1]) + max(0, lo - out[0])
    res = np.empty_like(out)
    res[order] = out
    return res

In [ ]:
# The model: Qwen2.5-7B-Instruct, open weights, 32K-token window
MODEL_NAME = "Qwen2.5-7B-Instruct"
found = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
found = [os.path.dirname(p) for p in found if "7b-instruct" in p.lower()]
MODEL_PATH = found[0] if found else "Qwen/Qwen2.5-7B-Instruct"
# bf16 needs Ampere or newer (L4); T4 only emulates it, so use fp16 there
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
if torch.cuda.get_device_capability(0)[0] < 8:
    # T4 has no flash kernel, and SDPA's memory-efficient kernel can't do grouped-query attention, so
    # by default a long prompt falls back to the math kernel and builds the full tokens × tokens matrix
    # (7 GB at 8K). Repeat the key/value heads instead, and never allow the math kernel.
    import transformers.integrations.sdpa_attention as sdpa_impl
    sdpa_impl.use_gqa_in_sdpa = lambda *args, **kwargs: False
    torch.backends.cuda.enable_math_sdp(False)
v = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
dtype_arg = {"dtype" if v >= (4, 56) else "torch_dtype": DTYPE}

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", attn_implementation="sdpa", **dtype_arg).eval()
BASE, HEAD = model.get_decoder(), model.get_output_embeddings()
STOP = {t for t in (tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>"),
                    tok.convert_tokens_to_ids("<|endoftext|>")) if isinstance(t, int) and t >= 0}
print(f"{MODEL_NAME} · {DTYPE} · from {MODEL_PATH}")

In [ ]:
# The haystack: ten public-domain novels from Project Gutenberg
BOOKS = [1342, 1661, 84, 2701, 98, 35, 345, 1400, 36, 120]
TITLES = {1342: "Pride and Prejudice", 1661: "The Adventures of Sherlock Holmes", 84: "Frankenstein",
          2701: "Moby Dick", 98: "A Tale of Two Cities", 35: "The Time Machine", 345: "Dracula",
          1400: "Great Expectations", 36: "The War of the Worlds", 120: "Treasure Island"}

def load_book(n):
    hit = glob.glob(f"/kaggle/input/**/pg{n}.txt", recursive=True)
    if hit:
        raw = open(hit[0], encoding="utf-8").read()
    else:
        import urllib.request
        url = f"https://www.gutenberg.org/cache/epub/{n}/pg{n}.txt"
        raw = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    raw = raw.replace("\r\n", "\n")
    s = re.search(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\n", raw)
    e = re.search(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK", raw)
    body = raw[s.end() if s else 0: e.start() if e else None]
    body = re.sub(r"\[Illustration[^\]]*\]", "", body)
    hits = re.finditer(r"\n\s*(?:CHAPTER|Chapter) (?:1|I|ONE)\b", body[: len(body) // 6])
    # Skip table-of-contents entries: the real heading is followed by prose, not another heading
    ch = next((m for m in hits if not re.search(r"(?:CHAPTER|Chapter) ", body[m.end(): m.end() + 300])), None)
    body = body[ch.start():] if ch else body[len(body) // 50:]
    return re.sub(r"[ \t]+", " ", re.sub(r"\n{3,}", "\n\n", body)).strip()

TEXT = {n: load_book(n) for n in BOOKS}
TOKS = {n: tok(TEXT[n], add_special_tokens=False).input_ids for n in BOOKS}
for n in BOOKS:
    print(f"{TITLES[n]:<36} {len(TOKS[n]):>8,} tokens")

In [ ]:
# The facts we hide ("needles"). Fact i always lives in book i.
# Find the words: the question repeats the fact's own words.
WORDS = [dict(fact=n["statement"], q=n["question"], a=n["answer"],
              rule="Answer with just the answer from the story.") for n in [
 {
  "statement": "The Wi-Fi password at the venue is blue-dorayaki-42.",
  "question": "What is the Wi-Fi password at the venue?",
  "answer": "blue-dorayaki-42"
 },
 {
  "statement": "The secret code for free chai at the canteen is MASALA-7781.",
  "question": "What is the secret code for free chai at the canteen?",
  "answer": "MASALA-7781"
 },
 {
  "statement": "The hackathon demo room is Room 4C-17.",
  "question": "Which room is the hackathon demo room?",
  "answer": "4C-17"
 },
 {
  "statement": "The locker combination for the robotics lab is 38-11-52.",
  "question": "What is the locker combination for the robotics lab?",
  "answer": "38-11-52"
 },
 {
  "statement": "Priya's pet tortoise is named Captain Pakora.",
  "question": "What is the name of Priya's pet tortoise?",
  "answer": "Captain Pakora"
 },
 {
  "statement": "The admin port of the demo server is 47291.",
  "question": "What is the admin port of the demo server?",
  "answer": "47291"
 },
 {
  "statement": "The cricket team's lucky jersey number this season is 863.",
  "question": "What is the cricket team's lucky jersey number this season?",
  "answer": "863"
 },
 {
  "statement": "The secret ingredient in Ravi's biryani is smoked kokum.",
  "question": "What is the secret ingredient in Ravi's biryani?",
  "answer": "smoked kokum"
 },
 {
  "statement": "The campus shuttle leaves from Gate 14B.",
  "question": "Which gate does the campus shuttle leave from?",
  "answer": "14B"
 },
 {
  "statement": "The DJ night at the fest starts at 9:40 pm.",
  "question": "What time does the DJ night at the fest start?",
  "answer": "9:40"
 }
]]

# Find the meaning: no shared words. You have to know where the landmark is.
MEANING = [dict(fact=f, q=f"Which character has been to {city}?", a=name,
                rule="Answer with just the character's name.") for f, city, name in [
 [
  "Kabir's flat looks straight out at the Charminar.",
  "Hyderabad",
  "Kabir"
 ],
 [
  "Ananya got married in a hall right beside the Hawa Mahal.",
  "Jaipur",
  "Ananya"
 ],
 [
  "Rohan runs a flower stall at the east gate of the Meenakshi Temple.",
  "Madurai",
  "Rohan"
 ],
 [
  "Tara's office window faces the Burj Khalifa.",
  "Dubai",
  "Tara"
 ],
 [
  "Arjun proposed to his wife on the rooftop of the Marina Bay Sands.",
  "Singapore",
  "Arjun"
 ],
 [
  "Zoya grew up a short walk from the Golden Temple.",
  "Amritsar",
  "Zoya"
 ],
 [
  "Vikram jogs around the Qutub Minar every morning.",
  "Delhi",
  "Vikram"
 ],
 [
  "Meera lives right next to the Gateway of India.",
  "Mumbai",
  "Meera"
 ],
 [
  "Ishaan sketches the domes of the Hagia Sophia every weekend.",
  "Istanbul",
  "Ishaan"
 ],
 [
  "Neha hikes up Table Mountain before work.",
  "Cape Town",
  "Neha"
 ]
]]

for w, m in zip(WORDS[:3], MEANING[:3]):
    print(f"words:   {w['q']:<55} ← {w['fact']}")
    print(f"meaning: {m['q']:<55} ← {m['fact']}\n")

In [ ]:
# Build a prompt: the first N tokens of a novel, the fact at a chosen depth, the question at the end
def haystack(book, n_tokens):
    text = tok.decode(TOKS[book][:n_tokens])
    cut = max(text.rfind(". "), text.rfind(".\n"), text.rfind('." '))
    return text[: cut + 1] if cut > 0 else text

def insert(text, fact, depth):
    spots = [0] + [m.end() for m in re.finditer(r'[.!?]["”’]?\s+', text)] + [len(text)]
    at = min(spots, key=lambda p: abs(p - depth * len(text)))
    pad = " " if at and not text[at - 1].isspace() else ""
    return text[:at] + pad + fact + " " + text[at:]

def prompt(doc, needle, before=False):
    q = f"Question: {needle['q']}\n"
    user = ((q + "\n") if before else "") + (
        "Read the story below and answer the question after it.\n\n"
        f"<story>\n{doc}\n</story>\n\n{q}{needle['rule']}")
    return tok.apply_chat_template([{"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def ask(text, max_new=24):
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    out = BASE(input_ids=ids, use_cache=True)
    past, got = out.past_key_values, []
    nxt = HEAD(out.last_hidden_state[:, -1:]).argmax(-1)
    for _ in range(max_new):
        if nxt.item() in STOP:
            break
        got.append(nxt.item())
        out = BASE(input_ids=nxt, past_key_values=past, use_cache=True)
        past = out.past_key_values
        nxt = HEAD(out.last_hidden_state[:, -1:]).argmax(-1)
    del out, past
    return tok.decode(got, skip_special_tokens=True).strip(), ids.shape[1]

def right(reply, answer):
    return re.search(r"(?<![\w-])" + re.escape(answer.lower()) + r"(?![\w-])", reply.lower()) is not None

example = prompt(insert(haystack(BOOKS[0], 300), MEANING[0]["fact"], 0.5), MEANING[0])
print(example[example.find("<story>"):])

## Run it
10 facts × 2 kinds × 6 lengths = 120 questions.

In [ ]:
# Run: the same 20 questions at six prompt lengths (fact i keeps the same depth at every length)
LENGTHS = [1024, 2048, 4096, 8192, 16384, 31744]
rng = random.Random(7)
DEPTH = [round(rng.uniform(0.2, 0.8), 3) for _ in BOOKS]

rows = []
for L in LENGTHS:
    t0 = time.time()
    for kind, needles in (("words", WORDS), ("meaning", MEANING)):
        for i, nd in enumerate(needles):
            doc = insert(haystack(BOOKS[i], L), nd["fact"], DEPTH[i])
            reply, n = ask(prompt(doc, nd))
            rows.append(dict(kind=kind, length=L, tokens=n, needle=i, depth=DEPTH[i],
                             q=nd["q"], answer=nd["a"], reply=reply, ok=right(reply, nd["a"])))
    torch.cuda.empty_cache()
    done = [r for r in rows if r["length"] == L]
    w = np.mean([r["ok"] for r in done if r["kind"] == "words"]) * 100
    m = np.mean([r["ok"] for r in done if r["kind"] == "meaning"]) * 100
    print(f"{LABEL[L]:>4} tokens · words {w:5.0f}% · meaning {m:5.0f}% · {time.time() - t0:5.0f}s")

df = pd.DataFrame(rows)
df.to_json("dumb_zone_results.json", orient="records", indent=1)

## The result

In [ ]:
# The chart
acc = df.groupby(["kind", "length"]).ok.mean().unstack(0) * 100
xs = np.arange(len(LENGTHS))
lines = (("words", BLUE, "Find the words"), ("meaning", RED, "Find the meaning"))
starts = spread([acc[k].iloc[0] for k, _, _ in lines], 9, lo=0, hi=104)
ends = spread([acc[k].iloc[-1] for k, _, _ in lines], 9, lo=0, hi=104)
fig, ax = plt.subplots(figsize=(15, 7.5))
for (kind, color, name), y0, y1 in zip(lines, starts, ends):
    y = acc[kind].values
    ax.plot(xs, y, color=color, lw=5, marker="o", ms=13, zorder=3)
    ax.annotate(f"{y[0]:.0f}%", (xs[0], y0), xytext=(-18, 0), textcoords="offset points", ha="right",
                va="center", color=color, fontsize=20, fontweight="bold", annotation_clip=False)
    ax.annotate(f"{y[-1]:.0f}%  {name}", (xs[-1], y1), xytext=(20, 0), textcoords="offset points",
                va="center", color=color, fontsize=21, fontweight="bold", annotation_clip=False)
ax.set_xticks(xs, [LABEL[L] for L in LENGTHS])
ax.set_xlim(-0.8, len(xs) + 1.9)
ax.set_ylim(-4, 112)
ax.set_yticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"])
ax.grid(axis="y", alpha=0.7)
ax.set_xlabel("Tokens of novel around the fact")
ax.set_ylabel("Answered correctly")
ax.set_title("Same fact. Same question. Only more text around it.", loc="left", pad=22)
fig.text(0.01, 0.01, f"{MODEL_NAME} · {GPU} · {df.needle.nunique()} facts per point, each in its own novel · "
         f"greedy decoding · measured {TODAY}", color=MUTED, fontsize=12)
plt.tight_layout(rect=(0, 0.035, 1, 1))
plt.savefig("dumb_zone.png", dpi=160)
plt.show()

w, m = acc["words"], acc["meaning"]
print(f"Find the words:   {w.iloc[0]:.0f}% at {LABEL[LENGTHS[0]]} → {w.iloc[-1]:.0f}% at {LABEL[LENGTHS[-1]]}")
print(f"Find the meaning: {m.iloc[0]:.0f}% at {LABEL[LENGTHS[0]]} → {m.iloc[-1]:.0f}% at {LABEL[LENGTHS[-1]]}")

## What it said instead

In [ ]:
# What it said at 32K when it had to find the meaning
last = df[(df.length == LENGTHS[-1]) & (df.kind == "meaning")]
for _, r in last.iterrows():
    print(f"{'✓' if r.ok else '✗'}  {r.q:<44} expected {r.answer:<7} → {r.reply[:70]!r}")

## Read the chart
- **Blue** is what most "needle in a haystack" scores measure: can it find the exact words?
- **Red** is real work: can it connect two things that don't share words?
- The window never filled up: 32K is the model's limit, so every point on this chart is *inside* the window.

Sources: NoLiMa, Modarressi et al., ICML 2025 · Context Rot, Chroma, 2025.